In [2]:
from pathlib import Path

from clearml import Dataset
import pandas as pd
from tqdm.notebook import tqdm

In [3]:
def time_delta_stats(df, ts_col='_time', use_index=True, drop_duplicates=False):
    """
    Returns (stats_series, df_with_flags)

    stats_series: min, max, median, mean time deltas between consecutive timestamps
                  plus the same values in seconds (suffix '_sec').

    df_with_flags: original df with two extra columns:
        - 'delta' (Timedelta to previous timestamp after sorting by time)
        - 'delta_gt_mean' (bool: delta > global mean delta)
    """
    import pandas as pd

    # --- 1) Build a per-row datetime Series (UTC) ---
    if use_index:
        s_raw = pd.Series(df.index, index=df.index)
        if not isinstance(df.index, pd.DatetimeIndex):
            s_raw = pd.to_datetime(s_raw, utc=True, errors='coerce')
    else:
        s_raw = pd.to_datetime(df[ts_col], utc=True, errors='coerce')
        s_raw.index = df.index  # align to df rows

    # Series used for stats: drop NaT, sort, optional de-dup
    s = s_raw.dropna().sort_values()
    if drop_duplicates:
        s = s[~s.duplicated(keep='first')]

    # --- 2) Consecutive diffs on the cleaned series (stats basis) ---
    d = s.diff()  # first one is NaT

    # --- 3) Stats as timedeltas ---
    stats_td = d.dropna().agg(['min', 'max', 'median', 'mean'])

    # --- 4) Same stats in seconds ---
    stats_sec = stats_td / pd.Timedelta(seconds=1)
    stats_sec.index = [f'{k}_sec' for k in stats_sec.index]

    # --- 5) Combine stats ---
    out_stats = pd.concat([stats_td, stats_sec])

    # --- 6) Per-row delta back on the original df ---
    # Map each timestamp value to the delta computed for its (kept) occurrence.
    # (If drop_duplicates=True, duplicates share the same delta as the first occurrence.)
    # Build a mapping: timestamp -> delta (from the d Series computed above)
    # Note: s has unique timestamps when drop_duplicates=True.
    delta_by_ts = d.groupby(s).last()  # index: timestamps, values: delta

    # Now map back to all rows using their raw timestamps
    delta_per_row = s_raw.map(delta_by_ts)  # Timedelta per row; NaT where ts was NaT or earliest ts

    # --- 7) Flag rows whose delta > global mean (NaT -> False) ---
    mean_delta = stats_td['mean'] if not stats_td.empty else pd.Timedelta(0)
    delta_gt_mean = delta_per_row.gt(mean_delta).fillna(False)

    # --- 8) Return the augmented DataFrame copy ---
    df_out = df.copy()
    df_out['delta'] = delta_per_row
    df_out['delta_gt_mean'] = delta_gt_mean

    return out_stats, df_out


# Params

In [4]:
freq = "10s"


# Total Power

In [12]:
data_root = Dataset.get(
    dataset_project="ForeSightNEXT/Electric Load Forecasting",
    dataset_name="household-1235",
    dataset_version="2.0.0",
).get_local_copy()

raw_data_path = Path(data_root, "uri:urn:554b7c6c19254be5/power.csv")
resampled_data_path = Path(f"../data/resampled-{freq}/total_power.csv")

In [6]:
def cumsum_total_resample(df, freq):
    df_copy = df.copy()
    df_copy["total"] = df_copy['total'].fillna(0).cumsum().where(df['total'].notna())
    df_copy = df_copy.resample(freq).mean()
    return df_copy

In [13]:
import os


if resampled_data_path.exists() and resampled_data_path.is_file():
    print("Resampled File already exists")
    resampled_data = pd.read_csv(
        resampled_data_path,
        index_col="_time",
        parse_dates=True,
        date_format="ISO8601",
    )
else:
    data = pd.read_csv(
        raw_data_path,
        usecols=["_time", "_value"],
        index_col="_time",
        parse_dates=True,
        date_format="ISO8601",
    )
    data.rename(columns={"_value": "total"}, inplace=True)
    display(data)

    # os.makedirs(f"../data/resampled-{freq}", exist_ok=True)
    # cumsum_total_resample(data, freq).to_csv(f"../data/resampled-{freq}/total_power_cumsum.csv")

    # data.to_csv(Path(f"../data/total_power_raw.csv"))

    resampled_data_path.parent.mkdir(exist_ok=True, parents=True)
    resampled_data = data.resample(freq).max() # !!! cahnge to max
    resampled_data.to_csv(resampled_data_path)

# display(resampled_data)

,total
_time,
2020-07-28 14:01:02.030000+00:00,331
2020-07-28 14:01:04.791000+00:00,331
2020-07-28 14:01:07.733000+00:00,333
2020-07-28 14:01:10.752000+00:00,330
2020-07-28 14:01:13.731000+00:00,332
...,...
2023-01-30 13:38:05.180000+00:00,440
2023-01-30 13:38:08.224000+00:00,437
2023-01-30 13:38:11.189000+00:00,448


# Devices

In [8]:
shiftable = {
    "DECT210 Waschmaschine": "uri:urn:6a112240-117e-4ec3-a129-5bc90908aedb",    
    "DECT200 Waschmaschine": "uri:urn:4b29c04c920141e8",
    "Smart Switch 6 Spülmaschine": "uri:urn:e91f9319-71af-4ddb-ab7d-fb47b45d69ad",    
    "DECT200 Spülmaschine": "uri:urn:cc256ae649904024",
    # "Smart Switch 6 Thermomix": "uri:urn:5b59894b-805d-4d7c-96c9-8ab42a18c3c7",
}

ts_dirs = {k: Path(data_root, fn) for k, fn in shiftable.items()}

In [9]:
for k, td in ts_dirs.items():
    print(k)
    display([t.name for t in td.iterdir()])
    print("")

DECT210 Waschmaschine


['sensor_160-value.csv', 'socket-on.csv', 'sensor_163-value.csv']


DECT200 Waschmaschine


['power.csv', 'energy.csv', 'temperature.csv', 'onOff.csv']


Smart Switch 6 Spülmaschine


['socket-on.csv', 'sensor_582-value.csv']


DECT200 Spülmaschine


['power.csv', 'energy.csv', 'temperature.csv', 'onOff.csv']

In [ ]:
def clean_cols(thing_name: str, data: pd.DataFrame):
    """ CAUTION: Side effects!
    """
    match thing_name:
        case "DECT210 Waschmaschine":
            data.rename(columns={"sensor_160-value": "power"}, inplace=True)
            data.drop(["sensor_163-value"], axis=1, inplace=True)
        case "Smart Switch 6 Spülmaschine":
            data.rename(columns={"sensor_582-value": "power"}, inplace=True)
    return data  

def read_tables(thing_name: str, freq: str, **kwargs):
    frames = []
    for p in tqdm(
        list(ts_dirs[thing_name].iterdir())# + 
    ):
        if p.suffix == ".csv" and any([
            p.stem == "energy",
            p.stem == "power",
            p.stem.startswith("sensor"),
        ]):
            df = pd.read_csv(
                p,
                usecols=["_time", "_value"],
                index_col="_time",
                parse_dates=True,
                date_format="ISO8601",
                **kwargs,
            )
            df.columns = [p.stem]
            df.astype(float)
            df = df.resample(freq).max() # !!! changed to max

            df_process = df.copy()
            bins_per_hour = int(pd.Timedelta("1h") / pd.Timedelta("10s"))  # 360
            df_process = df_process.ffill(limit=bins_per_hour) 

            frames.append(df_process)
    bulk = pd.concat(frames, axis=1)
    bulk = clean_cols(thing_name, bulk)
    bulk.to_csv(f'../data/things/{thing_name}.csv')
    return bulk

def cached_resample(thing_name: str, freq=str):
    out_path = Path(f"../data/resampled-{freq}/things/{thing_name}.csv")
    if out_path.exists() and out_path.is_file():
        print("reading from cache...")
        time_series = pd.read_csv(
            out_path, 
            index_col="_time",
            parse_dates=True,
            date_format="ISO8601"
        )
    else:
        print("resampling...")
        time_series = read_tables(thing_name=thing_name, freq=freq)
        out_path.parent.mkdir(parents=True, exist_ok=True)   
        time_series.to_csv(out_path)

    return time_series

In [11]:
series = {thing_name: cached_resample(thing_name, freq) for thing_name in ts_dirs.keys()}

resampling...


  0%|          | 0/3 [00:00<?, ?it/s]

resampling...


  0%|          | 0/4 [00:00<?, ?it/s]

resampling...


  0%|          | 0/2 [00:00<?, ?it/s]

resampling...


  0%|          | 0/4 [00:00<?, ?it/s]